In [1]:
import numpy as np
import pandas as pd
import re
from pathlib import Path

In [2]:
DATA = Path("../data")

lyrics_df = pd.read_csv(DATA / 'lyrics_transformed.csv').drop('Unnamed: 0', axis=1)
audio_df = pd.read_csv(DATA / 'audio_features_transformed.csv').drop('Unnamed: 0', axis=1)

Currently, the formatting of song names in the dataframes are different. We make these consistent with a function.

In [3]:
def transform_name(name):
    #get rid of spaces
    name = re.sub(r" ", "", name)

    #get rid of apostrophes and +
    name = re.sub(r"['’‘+\"]", "", name)

    #change punctuation to underscore
    name = re.sub(r'[()?.,\-!&]', "_", name)

    #lowercase all
    name = name.lower()

    #remove 'bonus track'
    name = re.sub(r"_bonustrack", "", name)
    
    #remove features
    name = re.sub(r'_feat_[a-z]+_', "", name)

    #final exceptions
    name = re.sub(r"popversion_", "popversion", name)
    name = re.sub(r"atthedisco_", "", name)
    
    return name

def normalise_apostrophes(name):
    return re.sub(r"[\"’‘]", "'", name)

list_of_names = [normalise_apostrophes(name) for name in audio_df['name']]
list_of_names.pop(list_of_names.index('Teardrops On My Guitar - Radio Single Remix'))

names_df = pd.DataFrame({'raw': list_of_names, 'name': [transform_name(name) for name in list_of_names]})

audio_df['name'] = audio_df['name'].apply(transform_name)
lyrics_df['name'] = lyrics_df['name'].apply(transform_name)
df = audio_df.merge(lyrics_df, on='name')
df['id'] = [i for i in range(len(df))]

id_df = df[['id', 'name']]
names_df = names_df.merge(id_df, on='name').drop('name', axis=1)

order = [(names_df['raw'][i], i) for i in range(len(names_df))]
name_to_id = dict(order)
id_to_name = dict([(j, i) for (i, j) in order])


In [4]:
lyrics_df = df[lyrics_df.columns].drop('name', axis=1)
audio_df = df[audio_df.columns].drop(['name', 'album'], axis=1)

audio_df.head()

,right_skewed__liveness,right_skewed__speechiness,right_skewed__duration_s,remainder__acousticness,remainder__danceability,remainder__tempo,remainder__valence,remainder__popularity
0,-0.590843,-0.862645,-0.057864,0.490892,-0.651337,2.112395,-0.552443,1.830861
1,0.128932,-1.311837,1.257538,-0.895145,0.220610,-0.440299,-0.492848,1.435663
2,1.793908,-1.186914,-0.740315,-0.624170,0.150854,-0.852065,0.531117,1.567396
3,-0.633483,1.190649,0.662855,0.668080,-0.328716,1.103840,-1.164655,1.830861
4,-1.034082,2.196511,0.703308,1.187424,-1.357613,1.119797,-0.731231,1.567396


Dissimilarity matrices can be calculated. Euclidean distance is used for audio features for consistency with original project. Cosine similarity is used for lyrics for better accuracy with semantics.

In [5]:
from sklearn.metrics import pairwise_distances

D_lyrics = pairwise_distances(lyrics_df, metric='cosine')
D_audio = pairwise_distances(audio_df, metric='euclidean')

# This needs scaling - dividing both by max will scale distances to [0, 1], since they are non-negative
audio_max, lyrics_max = np.max(D_audio), np.max(D_lyrics)
np.max(D_audio), np.max(D_lyrics)

(np.float64(8.667975452557082), np.float64(1.5712285895235203))

In [ ]:
# convert matrices to DataFrames and scale
D_audio_df = pd.DataFrame(D_audio, columns=[i for i in range(len(df))])
D_lyrics_df = pd.DataFrame(D_lyrics, columns=[i for i in range(len(df))])

D_audio_df = D_audio_df/audio_max
D_lyrics_df = D_lyrics_df/lyrics_max

# for streamlit deployment
#D_audio_df.to_csv('audio_dissimilarities.csv', index=False)
#D_lyrics_df.to_csv('lyrics_dissimilarities.csv', index=False)

D_audio_df.head()

,0,1,2,3,4,5,6,7,8,9,...,227,228,229,230,231,232,233,234,235,236
0,0.000000,0.396425,0.491853,0.288758,0.403041,0.572610,0.436796,0.480232,0.402396,0.387742,...,0.632782,0.465504,0.553487,0.453771,0.633366,0.611473,0.535236,0.585507,0.503076,0.519160
1,0.396425,0.000000,0.328196,0.414916,0.556929,0.412791,0.430403,0.501950,0.275410,0.227174,...,0.504571,0.485983,0.444993,0.429252,0.472370,0.432847,0.517576,0.498324,0.490466,0.660430
2,0.491853,0.328196,0.000000,0.543384,0.658384,0.506448,0.420690,0.422330,0.365586,0.460761,...,0.410142,0.608700,0.471947,0.439762,0.427137,0.260575,0.537130,0.483814,0.539993,0.666275
3,0.288758,0.414916,0.543384,0.000000,0.191654,0.391695,0.421308,0.382292,0.426502,0.260782,...,0.682694,0.502515,0.620388,0.547347,0.652732,0.637867,0.515023,0.613984,0.544496,0.584895
4,0.403041,0.556929,0.658384,0.191654,0.000000,0.413394,0.437545,0.431608,0.531795,0.391939,...,0.753877,0.551018,0.697879,0.618902,0.703486,0.733797,0.544497,0.694312,0.602710,0.595409


Recommendation algorithm 1: k-NN approach. Calculate a convex combination of dissimilarities, and return the k smallest dissimilarities, ie, $$D_{combined} := \alpha D_{audio} + (1-\alpha) D_{lyrics}$$ for some $\alpha$.

Different recommendations could be offered by varying $\alpha$, for example, 0.3 for lyric-heavy, 0.5 for balanced, 0.7 for music-heavy.


In [7]:
def recommend(query_name, alpha=0.5, k=5):
    D = alpha * D_audio_df + (1 - alpha) * D_lyrics_df
    query_id = name_to_id[query_name]
    distances = D.loc[query_id].copy()
    distances.loc[query_id] = np.inf  # don't recommend itself
    return [id_to_name[x] for x in distances.nsmallest(k).index]

recs = []
alphas = []
for alpha in np.arange(0, 1.1, 0.1):
    alphas.append(alpha)
    recs.append(recommend("Style (Taylor's Version)", alpha=alpha))

alpha_df = pd.DataFrame({
    'alpha': alphas, 
    'rec_1': [recs[i][0] for i in range(len(recs))],
    'rec_2': [recs[i][1] for i in range(len(recs))],
    'rec_3': [recs[i][2] for i in range(len(recs))],
    'rec_4': [recs[i][3] for i in range(len(recs))],
    'rec_5': [recs[i][4] for i in range(len(recs))]
})
alpha_df

,alpha,rec_1,rec_2,rec_3,rec_4,rec_5
0,0.0,Gorgeous,I Wish You Would (Taylor's Version),Bad Blood (Taylor's Version),I Forgot That You Existed,So High School
1,0.1,I Wish You Would (Taylor's Version),Gorgeous,Bad Blood (Taylor's Version),I Forgot That You Existed,So High School
2,0.2,I Wish You Would (Taylor's Version),Gorgeous,Bad Blood (Taylor's Version),I Forgot That You Existed,So High School
3,0.3,I Wish You Would (Taylor's Version),Gorgeous,Bad Blood (Taylor's Version),We Are Never Ever Getting Back Together (Taylo...,Mr. Perfectly Fine (Taylor's Version) (From Th...
4,0.4,I Wish You Would (Taylor's Version),Bad Blood (Taylor's Version),Gorgeous,We Are Never Ever Getting Back Together (Taylo...,New Romantics (Taylor's Version)
5,0.5,I Wish You Would (Taylor's Version),Bad Blood (Taylor's Version),We Are Never Ever Getting Back Together (Taylo...,Gorgeous,'Slut!' (Taylor's Version) (From The Vault)
6,0.6,I Wish You Would (Taylor's Version),Bad Blood (Taylor's Version),We Are Never Ever Getting Back Together (Taylo...,Gorgeous,Suburban Legends (Taylor's Version) (From The ...
7,0.7,We Are Never Ever Getting Back Together (Taylo...,I Wish You Would (Taylor's Version),Bad Blood (Taylor's Version),Suburban Legends (Taylor's Version) (From The ...,Superman (Taylor's Version)
8,0.8,Superman (Taylor's Version),We Are Never Ever Getting Back Together (Taylo...,Suburban Legends (Taylor's Version) (From The ...,The Last Time (feat. Gary Lightbody of Snow Pa...,I Wish You Would (Taylor's Version)
9,0.9,Superman (Taylor's Version),The Last Time (feat. Gary Lightbody of Snow Pa...,Suburban Legends (Taylor's Version) (From The ...,We Are Never Ever Getting Back Together (Taylo...,Bad Blood (feat. Kendrick Lamar) (Taylor's Ver...


Recommendation algorithm 2: Historical searches can also be considered in a similar k-NN approach. The current search should dominate the results, while previous searches can be weighted by considering recency.

Let $D_{current}$ be the dissimilarity row matrix of the current search, and $D_0, D_1, ..., D_{N-1}$ be the $N$ row matrices of ordered historical searches, most recent first. Let $W$ be scalar weights named similarly.

Then $$D_{combined} := W_{current} \cdot D_{current} + \sum_{i=0}^{N-1} \left(W(i, N) \cdot D_i \right)$$ where $W_{current} + \sum_{i} W(i, N) = 1$.

Let $W_{current} = 0.8$. Then the weight function $W(i, N) = \frac{2(N-i)}{5N(N+1)}$ satisfies $\sum_{i} W(i, N) = 0.2$, while also being a decreasing function of $i$ for constant $N$. 

In general, the weight function $W(i, N) = \frac{2(1-W_{current})(N-i)}{N(N+1)}$ is valid.

In [8]:
def recommend_weighted(query_name: str, prev_queries: list, alpha=0.5, k=5):
    """Takes query and a list of previous queries (ordered, most recent first)"""

    D = alpha * D_audio_df + (1 - alpha) * D_lyrics_df

    # weight function for current query
    if prev_queries:
        s_current = 0.8
    else:
        s_current = 1

    query_id = name_to_id[query_name]
    distances = D.loc[query_id].copy()*s_current
    distances.loc[query_id] = np.inf  # don't recommend itself

    # define weight function for previous queries
    def W(i, S):
        return (2*(S-i))/(5*S*(S+1))
        
    for i, query in enumerate(prev_queries):
        query_id = name_to_id[query]
        query_distances = D.loc[query_id].copy() * W(i, len(prev_queries))
        query_distances.loc[query_id] += 0.5/(i+1) # make it less likely to recommend something already searched 
        distances = distances + query_distances
    return [id_to_name[x] for x in distances.nsmallest(k).index]

recommend_weighted("Style (Taylor's Version)", ["The Archer", "champagne problems", "I Wish You Would (Taylor's Version)"], alpha=0.5)


["Bad Blood (Taylor's Version)",
 'Gorgeous',
 "We Are Never Ever Getting Back Together (Taylor's Version)",
 'Dear Reader',
 "Bad Blood (feat. Kendrick Lamar) (Taylor's Version)"]

Both recommendation algorithms can be combined into a single search function with a boolean flag.

In [9]:
def search(query_name: str, prev_queries: list, balance='equal', use_prev=True, k=5):
    """
    Returns top 5 recommendations given query name and list of previous queries. 
    balance takes values in {'equal', 'lyrics', 'audio'}, changing the fusion weights of the dissimilarity matrices.
    use_prev is boolean. True means previous searches are used to enhance recommendations.
    """

    balance_to_alpha = {
        'equal': 0.5,
        'lyrics': 0.3, 
        'audio': 0.7
    }

    assert type(use_prev) == bool
    if use_prev == True:
        return recommend_weighted(query_name, prev_queries, alpha=balance_to_alpha[balance], k=k)
    else:
        return recommend(query_name, alpha=balance_to_alpha[balance], k=k)
    
search('the last great american dynasty', ['Gorgeous', 'Lover'], balance='audio', use_prev=True, k=10)

['champagne problems',
 'tolerate it',
 'right where you left me - bonus track',
 "Wonderland (Taylor's Version)",
 'august',
 'Glitch',
 'High Infidelity',
 'Anti-Hero',
 'Maroon',
 'willow']

In [10]:
search('the last great american dynasty', ['Lover'], balance='audio', use_prev=True, k=10)

['tolerate it',
 'champagne problems',
 'right where you left me - bonus track',
 'august',
 "Wonderland (Taylor's Version)",
 'Glitch',
 'High Infidelity',
 'exile (feat. Bon Iver)',
 'cowboy like me',
 'Anti-Hero']